<a href="https://colab.research.google.com/github/Shatha-1/Data-Science/blob/main/Phase2_cleaned_openFDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import os
import pandas as pd

In [22]:
#Paths
RAW_PATH = "openfda_influenza_raw.json"
OUT_DIR = "phase2_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

#Load data
df = pd.read_json(RAW_PATH)

In [23]:
# Remove duplicate reports
if "safetyreportid" in df.columns:
    df = df.drop_duplicates(subset=["safetyreportid"])

#Flatten patient column (because it's a dict)
patient_df = pd.json_normalize(df["patient"])
patient_df.columns = ["patient_" + c for c in patient_df.columns]
df = pd.concat([df.drop(columns=["patient"]), patient_df], axis=1)

In [24]:
#Explode reactions (because it's a list)
df_exp = df.explode("patient_reaction").reset_index(drop=True)

#Flatten reaction dictionary
reaction_df = pd.json_normalize(df_exp["patient_reaction"])
reaction_df.columns = ["reaction_" + c for c in reaction_df.columns]
df_exp = pd.concat([df_exp.drop(columns=["patient_reaction"]), reaction_df], axis=1)

#Clean reaction text
df_exp["reaction_reactionmeddrapt"] = (
    df_exp["reaction_reactionmeddrapt"]
    .astype(str)
    .str.lower()
    .str.strip()
)

df_exp.loc[
    df_exp["reaction_reactionmeddrapt"].isin(["nan", "none", ""]),
    "reaction_reactionmeddrapt"
] = pd.NA


In [25]:
#Convert date
df_exp["receivedate"] = pd.to_datetime(
    df_exp["receivedate"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

#Clean age
df_exp["patient_patientonsetage"] = pd.to_numeric(
    df_exp["patient_patientonsetage"], errors="coerce"
)

df_exp.loc[
    (df_exp["patient_patientonsetage"] < 0) |
    (df_exp["patient_patientonsetage"] > 120),
    "patient_patientonsetage"
] = pd.NA


In [26]:
#Keep only needed columns
df_clean = df_exp[
    [
        "safetyreportid",
        "receivedate",
        "patient_patientonsetage",
        "patient_patientsex",
        "reaction_reactionmeddrapt"
    ]
].copy()

In [27]:
#Remove rows with no reaction
df_clean = df_clean.dropna(subset=["reaction_reactionmeddrapt"])

#Rename columns
df_clean = df_clean.rename(columns={
    "safetyreportid": "report_id",
    "receivedate": "report_date",
    "patient_patientonsetage": "patient_age",
    "patient_patientsex": "patient_sex",
    "reaction_reactionmeddrapt": "reaction"
})


In [28]:
#Save file
out_path = os.path.join(OUT_DIR, "openfda_phase2_cleaned.csv")
df_clean.to_csv(out_path, index=False)

print("Saved:", out_path)
print("Final shape:", df_clean.shape)
print("Missing age:", df_clean["patient_age"].isna().sum())
print("Missing sex:", df_clean["patient_sex"].isna().sum())

Saved: phase2_outputs/openfda_phase2_cleaned.csv
Final shape: (7566, 5)
Missing age: 1394
Missing sex: 387


In [29]:
#Check missing values (for verification)
print("Missing age:", df_clean["patient_age"].isna().sum())
print("Missing sex:", df_clean["patient_sex"].isna().sum())

print("Age missing %:", round(df_clean["patient_age"].isna().mean()*100, 2))
print("Sex missing %:", round(df_clean["patient_sex"].isna().mean()*100, 2))

Missing age: 1394
Missing sex: 387
Age missing %: 18.42
Sex missing %: 5.11
